# EPANET Import — Complex Topology Verification

Mirrors **`tests/test_complex_topology_from_inp.py`**: `load_inp()` on `tests/networks/complex_topology.inp`, pre-trip heads/flows vs **wntr/EPANET**, then Pump_A trip checks.

Requires **`wntr`** (`pip install wntr` or `pip install 'rthym-moc[inp]'`).

> **Related:** [`quickstart_notebook.ipynb`](quickstart_notebook.ipynb) (Joukowsky R-THYM case).

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook(require_wntr=True)
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Evaluate

In [ ]:
from complex_topology_verification_utils import evaluate_complex_topology, HEAD_NODES

bundle = evaluate_complex_topology()
head_pass = sum(1 for m in bundle.pretrip_head_metrics if m.passed)
flow_pass = sum(1 for m in bundle.pretrip_flow_metrics if m.passed)
print(f"Pre-trip heads: {head_pass}/{len(bundle.pretrip_head_metrics)} passed")
print(f"Pre-trip flows: {flow_pass}/{len(bundle.pretrip_flow_metrics)} passed")
for chk in bundle.trip_checks:
    print(f"  {chk.name}: PASS={chk.passed} ({chk.detail})")

## 3. Pre-trip head errors

In [ ]:
nodes = [m.id for m in bundle.pretrip_head_metrics]
errs = [m.error for m in bundle.pretrip_head_metrics]
colors = ["seagreen" if m.passed else "crimson" for m in bundle.pretrip_head_metrics]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(nodes)), errs, color=colors)
ax.axhline(0.5, color="gray", linestyle="--", label="tol 0.5 ft")
ax.set_xticks(range(len(nodes)), nodes, rotation=45, ha="right")
ax.set_ylabel("|sim − EPANET| (ft)")
ax.set_title("Pre-trip junction head error")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Pump trip at Junction_E

In [ ]:
t = bundle.time_s
h = np.asarray(bundle.node_head["Junction_E"])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, h, color="steelblue")
ax.axvline(10.0, color="orange", linestyle="--", label="Pump trip")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Junction_E head (ft)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()